# Electricity Price Index Forecasting Workflow

This notebook provides a detailed, reproducible workflow for:
- data loading and quality checks,
- feature engineering (time, lag, and rolling statistics),
- model training with time-series cross-validation,
- quantitative evaluation and visualization,
- interpretation and export of results.


## 1. Environment Setup

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Configuration

In [ ]:
DATA_PATH = Path('data/raw/electricity_price_index.csv')
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'price_index'
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Data path:', DATA_PATH)
print('Output dir:', OUTPUT_DIR.resolve())

## 3. Load and Validate Data

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing dataset at {DATA_PATH}. '
        'Create a CSV file with columns [timestamp, price_index] before running.'
    )

df = pd.read_csv(DATA_PATH)
required_cols = {TIMESTAMP_COL, TARGET_COL}
missing = required_cols.difference(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors='coerce')
df = df.dropna(subset=[TIMESTAMP_COL, TARGET_COL]).sort_values(TIMESTAMP_COL).reset_index(drop=True)

print('Rows:', len(df))
display(df.head())

## 4. Exploratory Data Analysis

In [ ]:
stats_table = df[TARGET_COL].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
display(stats_table.to_frame(name='value'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df[TIMESTAMP_COL], df[TARGET_COL], linewidth=1.0)
axes[0].set_title('Price Index Over Time')
axes[0].set_xlabel('Timestamp')
axes[0].set_ylabel('Price Index')

sns.histplot(df[TARGET_COL], kde=True, ax=axes[1], bins=40)
axes[1].set_title('Price Distribution')
axes[1].set_xlabel('Price Index')

plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
feat = df.copy()
feat['hour'] = feat[TIMESTAMP_COL].dt.hour
feat['dayofweek'] = feat[TIMESTAMP_COL].dt.dayofweek
feat['month'] = feat[TIMESTAMP_COL].dt.month
feat['quarter'] = feat[TIMESTAMP_COL].dt.quarter
feat['is_weekend'] = (feat['dayofweek'] >= 5).astype(int)

for lag in [1, 2, 3, 6, 12, 24]:
    feat[f'lag_{lag}'] = feat[TARGET_COL].shift(lag)

for w in [3, 6, 12, 24]:
    feat[f'roll_mean_{w}'] = feat[TARGET_COL].rolling(w).mean()
    feat[f'roll_std_{w}'] = feat[TARGET_COL].rolling(w).std()

feat = feat.dropna().reset_index(drop=True)
feature_cols = [c for c in feat.columns if c not in [TIMESTAMP_COL, TARGET_COL]]
X = feat[feature_cols]
y = feat[TARGET_COL]

print('Feature rows:', X.shape[0])
print('Feature columns:', X.shape[1])
display(X.head())

## 6. Model Training with Time-Series CV

In [ ]:
def eval_metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred, squared=False),
        'R2': r2_score(y_true, y_pred)
    }

models = {
    'LinearRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )
}

tscv = TimeSeriesSplit(n_splits=5)
records = []
plot_payload = {}

for fold, (tr_idx, te_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
    y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        m = eval_metrics(y_test, pred)
        m.update({'fold': fold, 'model': name})
        records.append(m)

        if fold == 5:
            plot_payload[name] = (y_test.values, pred)

results = pd.DataFrame(records)
display(results.head())

## 7. Results Summary

In [ ]:
summary = (
    results.groupby('model')[['MAE', 'RMSE', 'R2']]
    .agg(['mean', 'std'])
    .reset_index()
)

summary.columns = [
    'model',
    'MAE_mean', 'MAE_std',
    'RMSE_mean', 'RMSE_std',
    'R2_mean', 'R2_std'
]

summary = summary.sort_values('RMSE_mean')
display(summary)

summary.to_csv(OUTPUT_DIR / 'cv_summary_notebook.csv', index=False)
results.to_csv(OUTPUT_DIR / 'cv_fold_results_notebook.csv', index=False)
print('Saved notebook outputs to:', OUTPUT_DIR)

## 8. Forecast Comparison Plot (Last Fold)

In [ ]:
for model_name, (y_true, y_pred) in plot_payload.items():
    plt.figure(figsize=(12, 4))
    plt.plot(y_true, label='Actual', linewidth=2)
    plt.plot(y_pred, label='Predicted', linewidth=1.5)
    plt.title(f'Actual vs Predicted ({model_name}, Final CV Fold)')
    plt.xlabel('Test Samples (ordered in time)')
    plt.ylabel('Price Index')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Interpretation Notes

Use this section to document:
- which model performed best and why,
- whether the model handles spikes/volatility,
- how results align with hypotheses from the paper,
- what improvements to test next (feature enrichment, ensembles, probabilistic forecasts).
